# Explainable Phishing URL Detection

This notebook is a reproducible front end for the project's shared `features.py` and `train.py` pipeline. Run it from the project directory. Training recomputes features from raw URLs, removes duplicate URLs, selects a model using validation F1, and reports held-out test metrics. **It does not visit URLs.**

> Important: the expected label convention is `0 = legitimate`, `1 = phishing`; confirm the original dataset documentation before treating the labels as ground truth. The dataset has no timestamp column, so the adaptive experiment is a group-batch simulation rather than temporal-drift evidence.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from features import extract_features, feature_names
from train import load_and_validate, train

PROJECT = Path.cwd()
DATA_PATH = PROJECT / "dataset.csv"
ARTIFACTS = PROJECT / "artifacts"
assert DATA_PATH.exists(), f"Run this notebook from the project folder: {PROJECT}"

## 1. Dataset audit

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Rows:", len(raw))
print("Columns:", raw.columns.tolist())
print("Missing values by column (top):")
display(raw.isna().sum().sort_values(ascending=False).head(10).to_frame("missing"))
print("Duplicate URLs:", raw["url"].astype(str).duplicated().sum())
print("Label counts:")
display(raw["label"].value_counts(dropna=False).sort_index().rename(index={0:"Legitimate (assumed)", 1:"Phishing (assumed)"}).to_frame("rows"))
print("Label convention and source quality must be confirmed independently.")

## 2. Feature extraction consistency checks

In [ ]:
example_urls = [
    "https://example.com/path",
    "http://192.0.2.1/login?account=1",
    "secure-login-paypal.xyz/verify@account",
]
feature_preview = pd.DataFrame([extract_features(u) for u in example_urls], index=example_urls)
assert list(feature_preview.columns) == feature_names()
assert feature_preview.isna().sum().sum() == 0
feature_preview.T

## 3. Train models and generate evaluation evidence

In [ ]:
# This cell trains models and generates plots/metrics under artifacts/.
# It can take several minutes on the full 120,000-row dataset.
report = train(str(DATA_PATH))
print("Selected model:", report["selected_model"])
print("Split:", report["split_method"])
pd.DataFrame(report["validation_comparison"]).sort_values("Validation F1", ascending=False)

## 4. Held-out test results

In [ ]:
report = json.loads((ARTIFACTS / "metrics.json").read_text())
metrics = report["test_metrics"]
pd.DataFrame({"Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Average precision", "Brier score"],
              "Value": [metrics[k] for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision", "brier_score"]]})

## 5. Evaluation graphs

In [ ]:
from IPython.display import display, Image
plot_files = [
    "model_comparison.png", "roc_curves.png", "precision_recall_curves.png",
    "confusion_matrix.png", "calibration_curve.png", "threshold_analysis.png",
    "permutation_importance.png", "class_distribution.png", "adaptive_learning.png"
]
for name in plot_files:
    path = ARTIFACTS / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
    else:
        print(f"Not generated: {name} (check artifacts/ notes)")

## 6. Interpretation checklist

- Compare recall with precision: missed phishing and false alarms have different costs.
- Treat ROC-AUC and average precision as ranking metrics, not proof of deployment safety.
- Use the calibration curve before interpreting a predicted probability literally.
- Permutation importance is predictive association, not causal evidence.
- The adaptive-learning experiment uses deterministic domain batches because no reliable timestamps are available. It does not establish real-world temporal adaptation.
- Do not claim that SSL, DNS or server location were verified: those fields are string/TLD heuristics only.